In [ ]:
# Install necessary packages if running in Google Colab.
!pip install pymupdf
!pip install pypdf
!pip install orjsonl

In [ ]:
# Working with .pdf documents.
import fitz
import re

# Handling .JSONL objects.
import orjsonl

# Dealing with Optional typing
from typing import Optional

In [ ]:
# Extract plain text from .pdf file.
textbook_text = ""
textbook = fitz.open("textbook.pdf")
for page in textbook :
  textbook_text += page.get_text()

# Partition text into section containing problems and section containing
# solutions. This works specifically for the McMurray textbook.
textbook_sections = re.split("\nANSWER KEY.*\nChapter 1", textbook_text)
if len(textbook_sections) >= 2 :
    problems_text = textbook_sections[0]
    solutions_text = textbook_sections[1]
else :
    problems_text = "ERROR: Split not found."
    solutions_text = "ERROR: Split not found."

In [ ]:
# Define function for extracting the problems and solutions from textbook text.
def parse_text_by_pattern(plain_text: str, pattern: str) -> str:
    """
    Function which parses input text and removes undesired pattern, used to
    remove watermarks, headers, and footers.

    Precondition: pattern is a regular expression.
    """
    out = ""
    parsed_text_array = re.split(pattern, plain_text)
    for segment in parsed_text_array:
        out += segment
    return out

def parse_problem_solution(plain_text: str) -> str:
    """
    Function which parses both problems and solutions, removing the watermarks,
    headers, and footers found in the McMurray textbook.
    """
    out = parse_text_by_pattern(plain_text, "\nAccess for free at openstax.org")
    out = parse_text_by_pattern(out, "\n[1-9][0-9]* •.*\n.*")
    out = parse_text_by_pattern(out, "\nAnswer Key\n.*")
    out = parse_text_by_pattern(out, "\nChapter.*")
    return out

def find_all_problems_solutions(all_text: str) -> list[str]:
    """
    Function which will find all problems and/or solutions in a given block of
    text. It will keep the problem ID, but will remove the problem header.

    Preconditions: All problems in the text start with "PROBLEM", chapters and
    subchapters follow McMurray textbook patterns.
    """
    out = []
    # Split the textbook into chapters, using McMurray Chapter patterns,
    # apply similar process for subchapters and problems.
    chapters_array = re.split("(\nCHAPTER|\nAPPENDIX|\nINDEX)", all_text)
    for chapter in chapters_array:
        subchapters_array = re.split("[1-9][0-9]*\\.[1-9][0-9]*.*", chapter)
        for subchapter in subchapters_array:
            problems_solutions_array = re.split("PROBLEM", subchapter)
            # First entry of the problems array is the chapter text found before
            # the first problem/solution, so it is skipped.
            for problem_solution in problems_solutions_array[1:]:
                # Parse out unnecessary text from problem then append to output.
                out.append(parse_problem_solution(problem_solution))
    return out

In [ ]:
# Find the problems and solutions in McMurray textbook.
problems_array = find_all_problems_solutions(problems_text)
solutions_array = find_all_problems_solutions(solutions_text)

In [ ]:
# Match problems to solutions in a dict using solution ID.
def get_id_prompt_completion(problem_solution_text: str) -> Optional[list[str]]:
    """
    Function which takes a problem or solution text and returns a list of the
    problem ID and the corresponding prompt or completion.

    Preconditions: problem_solution_text follows the McMurray textbook format.
    """
    # McMurray format numbers problems by Chapter number + "-" + Problem number.
    problem_solution_id = re.search(
        "\n[1-9][0-9]*-[1-9][0-9]*", problem_solution_text)
    if problem_solution_id is not None:
        problem_solution = problem_solution_text[problem_solution_id.end() + 1:]
        problem_solution_id = problem_solution_id.group(0)
        problem_solution_id = problem_solution_id[1:] # Remove leading \n
        return [problem_solution_id, problem_solution]
    else:
        return None

# Match the prompts and completions using ID.
prompts_completions = {}
for problem in problems_array:
    id_and_prompt = get_id_prompt_completion(problem)
    if id_and_prompt is not None:
        question_id = id_and_prompt[0]
        question_prompt = id_and_prompt[1]
        prompts_completions[question_id] = [question_prompt]

for solution in solutions_array:
    id_and_completion = get_id_prompt_completion(solution)
    if id_and_completion is not None:
        question_id = id_and_completion[0]
        question_completion = id_and_completion[1]
        if question_id in prompts_completions:
            prompts_completions[question_id].append(question_completion)

In [ ]:
# Convert prompts_completions dict into a list of JSON objects and save as
# .JSONL file.
prompts_completions_json = []
for question_id in prompts_completions:
    # Check that a non-empty prompt and non-empty completion has been found.
    if (len(prompts_completions[question_id]) == 2 and
        prompts_completions[question_id][0] != '' and
        prompts_completions[question_id][1] != ''):
        json = {
            "prompt" : prompts_completions[question_id][0],
            "completion" : prompts_completions[question_id][1]
        }
        prompts_completions_json.append(json)

# Save to .JSONL file.
orjsonl.save('mcmurray_prompts_completions.jsonl', prompts_completions_json)

In [ ]:
# OPTIONAL: Print all JSONs to verify that extraction was completed correctly.
for json in prompts_completions_json:
    print(json)